# DataSage â€” Chat Session (Local SQLite)

**Workflow:**
1. Run **Cell 0** â€” setup env + init DB
2. Run **Cell 1** â€” load your Excel or CSV file
3. Run **Cell 2** â€” start the chat session (type commands just like the real app)

Utility cells below Cell 2 handle users and stacks directly.

---

In [1]:
# ── Cell 0: SETUP (always run this first) ──────────────────────────────
# autoreload: auto-detects any code changes without kernel restart
%load_ext autoreload
%autoreload 2

import os, sys
# Add project root to path so `app.*` and `local_db` imports work
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
TEST_DIR = os.path.abspath(os.getcwd())
if TEST_DIR not in sys.path:
    sys.path.insert(0, TEST_DIR)
# Must be set BEFORE importing any app.* modules
os.environ['USE_LOCAL_DB'] = '1'
from local_db import init_db
init_db()
print('Ready. USE_LOCAL_DB =', os.environ['USE_LOCAL_DB'])


Database initialised at: c:\projects\data-sage-backend\test\database\datasage_test.db
Ready. USE_LOCAL_DB = 1


## 1. Load Your File

In [2]:
# â”€â”€ Cell 1: Load Excel or CSV â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Drop your file into test/excel/ then provide the path.
# After this cell runs, Cell 2 (chat) will pick it up automatically.

file_path = input('File path (xlsx / xls / csv): ').strip()

if file_path.lower().endswith('.csv'):
    import pandas as pd
    _df            = pd.read_csv(file_path)
    _sheet_columns = _df.columns.tolist()
    _sheet_data    = _df.to_dict(orient='records')
    # Pipeline engine requires each row to have an integer ID field
    _sheet_data    = [{'ID': i, **r} for i, r in enumerate(_sheet_data)]
    _sheet_styles  = {}
else:
    from app.services.excel_parser import parse_excel
    with open(file_path, 'rb') as fh:
        _raw = fh.read()
    _sheets        = parse_excel(_raw)
    if not _sheets:
        print('No sheets found in file.')
    else:
        _s             = _sheets[0]
        _sheet_columns = _s.columns
        _sheet_data    = [dict(r) for r in _s.data]
        _sheet_styles  = _s.styles
        if len(_sheets) > 1:
            print(f'Note: {len(_sheets)} sheets found â€” using first sheet ({_s.sheetName})')

import pandas as pd
print(f'Loaded {len(_sheet_data)} rows  |  {len(_sheet_columns)} columns:')
print(f'  {_sheet_columns}')
print()
pd.set_option('display.max_columns', None)
pd.DataFrame(_sheet_data, columns=_sheet_columns).head(5)


Loaded 46 rows  |  8 columns:
  ['Product_ID', 'Product_Name', 'Opening_Stock', 'Purchase_Stock_in', 'Number_of_Units_Sold', 'Hand_In_Stock', 'Cost_Price_Per_Unit_USD', 'Cost_Price_Total_USD']



,Product_ID,Product_Name,Opening_Stock,Purchase_Stock_in,Number_of_Units_Sold,Hand_In_Stock,Cost_Price_Per_Unit_USD,Cost_Price_Total_USD
0,P101,Laptop,50,25,10,60,1200,72000
1,P102,Monitor,40,21,5,50,500,25000
2,P103,Keyboard,60,21,15,70,50,3500
3,P104,Headphones,30,21,3,37,100,3700
4,P105,Smartphone,70,30,20,80,900,72000


## 2. Chat With Your Data

In [ ]:
import pandas as pd
from app.services.llm_service import run_detect_compute_stage, generate_pipeline_operations
from app.services.pipeline_engine import execute_pipeline
from app.models.schemas import PipelineOperation
from local_db import create_stack

try:
    _data     = [dict(r) for r in _sheet_data]
    _columns  = list(_sheet_columns)
    _styles   = dict(_sheet_styles)   # CellStyle objects or {}
except NameError:
    raise RuntimeError('Run Cell 0 (setup) and Cell 1 (load file) first.')

_orig_data    = [dict(r) for r in _data]
_orig_columns = list(_columns)
_orig_styles  = dict(_styles)
_ops          = []   # PipelineOperation objects applied so far
_snapshots    = []   # [(data, columns, styles, ops)] for undo

def _show(n=10):
    df = pd.DataFrame(_data, columns=_columns)
    print(f'  {len(_data)} rows  x  {len(_columns)} cols')
    print(df.head(n).to_string(index=True))
    print()

def _divider(): print('  ' + '-' * 60)

_divider()
print(f'  DataSage Chat  |  {len(_data)} rows  |  {len(_columns)} columns')
print(f'  Columns: {_columns}')
_divider()
print()

while True:
    try:
        cmd = input('You: ').strip()
    except (EOFError, KeyboardInterrupt):
        print('\n  Session ended.')
        break

    if not cmd:
        continue

    lower = cmd.lower()

    if lower in ('quit', 'exit', 'q'):
        print('  Session ended.')
        break

    if lower.startswith('show'):
        parts = lower.split()
        n = int(parts[1]) if len(parts) > 1 and parts[1].isdigit() else 10
        _show(n)
        continue

    if lower == 'cols':
        for i, c in enumerate(_columns):
            print(f'  {i}: {c}')
        print()
        continue

    if lower == 'history':
        if not _ops:
            print('  No operations yet.\n')
        else:
            for i, op in enumerate(_ops, 1):
                print(f'  {i:2}. [{op.type}]  {op.label}')
            print()
        continue

    if lower == 'undo':
        if not _snapshots:
            print('  Nothing to undo.\n')
        else:
            _data, _columns, _styles, _ops = _snapshots.pop()
            print(f'  Undone. Data restored to {len(_data)} rows.\n')
        continue

    if lower == 'reset':
        _snapshots.clear()
        _data    = [dict(r) for r in _orig_data]
        _columns = list(_orig_columns)
        _styles  = dict(_orig_styles)
        _ops     = []
        print(f'  Reset to original {len(_data)} rows.\n')
        continue

    if lower.startswith('save'):
        parts = cmd.split(maxsplit=1)
        name  = parts[1].strip() if len(parts) > 1 else 'Untitled Stack'
        uid   = input('  User ID: ').strip()
        ops_list = [op.dict() for op in _ops]
        stack, rows = create_stack(uid, name, ops_list)
        if rows:
            print(f'  Saved "{name}"  â€”  stack id: {stack["id"][:8]}...\n')
        else:
            print('  Save failed (user not found?).\n')
        continue

    print(f'  {cmd}\n')
    print('  [thinking...]\n')

    detect = await run_detect_compute_stage(cmd, _columns, _data[:20])

    if detect.get('exitEarly'):
        for cr in detect.get('computeResults', []):
            print(f'  {cr.label}: {cr.value}')
        print()
        continue

    ops, _ = await generate_pipeline_operations(
        detect['plannerInput'], _columns, _data[:20]
    )

    if not ops:
        print("  Couldn't generate operations try rephrasing.\n")
        continue

    # Intercept add_column: prompt user for default value (mirrors frontend UX)
    for _i, _op in enumerate(ops):
        if 'add_column' in str(_op.type):
            _col = _op.payload.get('column', '?')
            print(f'  New column: "{_col}"')
            _dflt = input('  Default value for all rows (Enter = empty): ').strip()
            _p = dict(_op.payload)
            _p['value'] = _dflt
            ops[_i] = PipelineOperation(id=_op.id, type=_op.type, label=_op.label, payload=_p)

    _snapshots.append(([dict(r) for r in _data], list(_columns), dict(_styles), list(_ops)))

    result   = execute_pipeline(_data, _columns, _styles, ops)
    prev_len = len(_data)
    _deleted = set(result.deletedRowIds)
    _data    = [r for r in result.transformedData if r.get("ID") not in _deleted]
    _columns = result.columns
    _styles  = result.cellStyles
    _ops.extend(ops)

    total_affected = sum(len(s.affectedRowIds) for s in result.steps)
    for step in result.steps:
        mark = 'OK' if step.status == 'success' else 'ERR'
        n_affected = len(step.affectedRowIds)
        if step.status == 'error':
            print(f'  [{mark}] [{step.operation.type}]  {step.errorMessage}')
        else:
            print(f'  [{mark}] [{step.operation.type}]  {step.summary}  ({n_affected} rows)')

    delta = len(_data) - prev_len
    delta_str = f'{delta:+d} rows' if delta != 0 else 'no row count change'
    print(f'\n  rows_affected = {total_affected}  |  {prev_len} -> {len(_data)} rows ({delta_str})')
    print()



  ------------------------------------------------------------
  DataSage Chat  |  46 rows  |  8 columns
  Columns: ['Product_ID', 'Product_Name', 'Opening_Stock', 'Purchase_Stock_in', 'Number_of_Units_Sold', 'Hand_In_Stock', 'Cost_Price_Per_Unit_USD', 'Cost_Price_Total_USD']
  ------------------------------------------------------------

  change the color of @Product_Name when the @Purchase_Stock_in is between median and average

  [thinking...]

  Couldn't generate operations try rephrasing.

  escape

  [thinking...]



---

## Utilities: Users & Stacks

These cells talk directly to the local SQLite DB â€” useful for setup and inspection.

In [ ]:
# â”€â”€ Cell 3: User Management â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from local_db import create_user, authenticate, list_users, delete_user
import sqlite3

print('Actions: create  login  list  delete')
action = input('Action: ').strip().lower()

if action == 'create':
    username = input('Username: ').strip()
    passcode = input('Passcode: ').strip()
    try:
        user, rows = create_user(username, passcode)
        print(f'rows_affected = {rows}')
        for k, v in user.items(): print(f'  {k}: {v}')
    except sqlite3.IntegrityError:
        print(f'Error: username already exists.')

elif action == 'login':
    username = input('Username: ').strip()
    passcode = input('Passcode: ').strip()
    user, rows = authenticate(username, passcode)
    print(f'rows_affected = {rows}')
    if user:
        print('OK:', user)
    else:
        print('Failed.')

elif action == 'list':
    users, count = list_users()
    print(f'rows_affected = {count}')
    for u in users:
        print(f"  {u['id'][:8]}  {u['username']}  {u['created_at']}")

elif action == 'delete':
    uid = input('User ID: ').strip()
    _, rows = delete_user(uid)
    print(f'rows_affected = {rows}')
    print('Deleted.' if rows else 'Not found.')

else:
    print('Unknown action.')


In [ ]:
# â”€â”€ Cell 4: Stack Management â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import json
from local_db import list_stacks, create_stack, get_stack, update_stack, delete_stack

print('Actions: list  get  create  update  delete')
action = input('Action: ').strip().lower()

if action == 'list':
    uid = input('User ID: ').strip()
    stacks, count = list_stacks(uid)
    print(f'rows_affected = {count}')
    for s in stacks:
        print(f"  {s['id'][:8]}  '{s['name']}'  ops={len(s['ops'])}  {s['created_at']}")

elif action == 'get':
    sid = input('Stack ID: ').strip()
    stack, rows = get_stack(sid)
    print(f'rows_affected = {rows}')
    if stack:
        for k, v in stack.items(): print(f'  {k}: {v}')
    else:
        print('Not found.')

elif action == 'create':
    uid  = input('User ID: ').strip()
    name = input('Stack name: ').strip()
    raw  = input('ops JSON (default []): ').strip() or '[]'
    stack, rows = create_stack(uid, name, json.loads(raw))
    print(f'rows_affected = {rows}')
    if stack:
        for k, v in stack.items(): print(f'  {k}: {v}')

elif action == 'update':
    sid  = input('Stack ID: ').strip()
    name = input('New name: ').strip()
    raw  = input('New ops JSON (default []): ').strip() or '[]'
    stack, rows = update_stack(sid, name, json.loads(raw))
    print(f'rows_affected = {rows}')
    if stack:
        for k, v in stack.items(): print(f'  {k}: {v}')
    else:
        print('Not found.')

elif action == 'delete':
    sid = input('Stack ID: ').strip()
    _, rows = delete_stack(sid)
    print(f'rows_affected = {rows}')
    print('Deleted.' if rows else 'Not found.')

else:
    print('Unknown action.')
